# 01 - Leakage-Window Ablation

Builds the reusable honest-evaluation core (`src/evaluation/metrics.py`, `src/features/windows.py`, `src/models/factory.py`) and proves empirically that including the contaminated months `N`/`N-1` inflates measured performance, establishing the honest modelling window `N-11..N-2`.

All logic lives in `src/`; this notebook only orchestrates calls, displays results, and writes the verification report. A single stratified train/test split (`test_size=0.3`, `random_state=42`, `stratify=y`) is reused across every window preset so the comparison is fair.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "README.md").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import OUTPUT_DIR, SEED, set_global_seed
from src.data.loaders import load_churn_panel
from src.features.windows import TARGET_COL, WINDOW_PRESETS, build_features
from src.models.factory import make_gbm
from src.evaluation.metrics import evaluate

set_global_seed()
df = load_churn_panel()
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

Loaded 99,807 rows x 55 columns


## 1. Single stratified split

The split is made once, at the row-index level, and reused across every window preset. Each window only changes which lag columns are visible to the model; the train/test partition of customers never changes, so differences in measured performance are attributable to the feature window alone.

In [2]:
y_all = df[TARGET_COL].to_numpy()
row_idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    row_idx, test_size=0.3, random_state=SEED, stratify=y_all
)

split_summary = {
    "method": "stratified train_test_split",
    "test_size": 0.3,
    "random_state": SEED,
    "n_train": int(len(idx_train)),
    "n_test": int(len(idx_test)),
    "train_positive_rate": float(y_all[idx_train].mean()),
    "test_positive_rate": float(y_all[idx_test].mean()),
}
split_summary

{'method': 'stratified train_test_split',
 'test_size': 0.3,
 'random_state': 42,
 'n_train': 69864,
 'n_test': 29943,
 'train_positive_rate': 0.038116912859269436,
 'test_positive_rate': 0.03810573422836723}

## 2. Window ablation

For each window preset, build features restricted to that window, fit a gradient boosting classifier on the fixed train partition, and evaluate on the fixed test partition.

In [3]:
def train_and_evaluate_window(df, months, idx_train, idx_test, y_all):
    """Build features for one window, fit on train, evaluate on test.

    The train/test indices are fixed once outside this function and reused
    across all window presets, so every window is compared on the same
    customer split; only the visible lag columns change.

    Args:
        df: The raw churn panel.
        months: Chronologically ordered month labels for this window.
        idx_train: Row indices assigned to the training split.
        idx_test: Row indices assigned to the test split.
        y_all: Full-length target array aligned with ``df``.

    Returns:
        Tuple of (metrics dict, fitted model, X_test, y_test).
    """
    X, _ = build_features(df, months)
    X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
    y_train, y_test = y_all[idx_train], y_all[idx_test]

    model = make_gbm()
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    metrics = evaluate(y_test, scores)
    return metrics, model, X_test, y_test

In [4]:
results = {}
fitted = {}
for name, months in WINDOW_PRESETS.items():
    metrics, model, X_test, y_test = train_and_evaluate_window(
        df, months, idx_train, idx_test, y_all
    )
    results[name] = metrics
    fitted[name] = (model, X_test, y_test)
    print(f"{name}: {metrics}")

results_df = pd.DataFrame(results).T
results_df.index.name = "window"
display(results_df)

ALL_12: {'roc_auc': 0.9741821080566941, 'pr_auc': 0.8296218151760325, 'brier': 0.012922097472287257, 'recall_at_10pct': 0.9255039439088519}


DROP_N: {'roc_auc': 0.8816088977899272, 'pr_auc': 0.5216989042463737, 'brier': 0.0244020636666198, 'recall_at_10pct': 0.6511831726555652}


HONEST: {'roc_auc': 0.8080642284250759, 'pr_auc': 0.37529689588363546, 'brier': 0.02866362220481329, 'recall_at_10pct': 0.5065731814198072}


DROP_N_N1_N2: {'roc_auc': 0.7882601972632999, 'pr_auc': 0.35430539395359195, 'brier': 0.029214174108089373, 'recall_at_10pct': 0.4890446976336547}


FAR: {'roc_auc': 0.7536063568231366, 'pr_auc': 0.316388972669411, 'brier': 0.030031936346675187, 'recall_at_10pct': 0.44084136722173534}


,roc_auc,pr_auc,brier,recall_at_10pct
window,,,,
ALL_12,0.974182,0.829622,0.012922,0.925504
DROP_N,0.881609,0.521699,0.024402,0.651183
HONEST,0.808064,0.375297,0.028664,0.506573
DROP_N_N1_N2,0.788260,0.354305,0.029214,0.489045
FAR,0.753606,0.316389,0.030032,0.440841


## 3. Relative PR-AUC inflation

Comparing the leaky `ALL_12` window (includes `N`, `N-1`) against the honest `HONEST` window (`N-11..N-2`).

In [5]:
pr_auc_all12 = results["ALL_12"]["pr_auc"]
pr_auc_honest = results["HONEST"]["pr_auc"]
relative_inflation = (pr_auc_all12 - pr_auc_honest) / pr_auc_all12

print(f"PR-AUC ALL_12:  {pr_auc_all12:.6f}")
print(f"PR-AUC HONEST:  {pr_auc_honest:.6f}")
print(f"Relative PR-AUC inflation (ALL_12 -> HONEST): {relative_inflation:.6f}")

PR-AUC ALL_12:  0.829622
PR-AUC HONEST:  0.375297
Relative PR-AUC inflation (ALL_12 -> HONEST): 0.547629


## 4. Permutation importance on the leaky (ALL_12) model

Computed on a fixed 6,000-row subsample of the ALL_12 test set, scored by average precision, to see which features the leaky model actually relies on.

In [6]:
leaky_model, leaky_X_test, leaky_y_test = fitted["ALL_12"]

rng = np.random.RandomState(SEED)
subsample_size = min(6000, len(leaky_X_test))
subsample_idx = rng.choice(len(leaky_X_test), size=subsample_size, replace=False)
X_perm = leaky_X_test.iloc[subsample_idx]
y_perm = leaky_y_test[subsample_idx]

perm_result = permutation_importance(
    leaky_model,
    X_perm,
    y_perm,
    scoring="average_precision",
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
)

importances = pd.DataFrame(
    {
        "feature": X_perm.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

top10_importances = importances.head(10).reset_index(drop=True)
display(top10_importances)

,feature,importance_mean,importance_std
0,USAGE_N,0.600347,0.015393
1,TAG_N-1,0.475089,0.007801
2,PAKET_SPEEDY_ID_11,0.263174,0.020932
3,STATUS_BAYAR_N,0.249114,0.014630
4,UMUR_PLG,0.055831,0.005223
5,USAGE_N-1,0.053359,0.005850
6,TAG_N,0.052036,0.007279
7,USAGE_N-3,0.035592,0.009210
8,WITEL_5,0.012087,0.003883
9,WITEL_3,0.009578,0.000686


## 5. Interpretation

Months `N` and `N-1` record the churn event rather than predict it: `USAGE_N` collapsing to zero and `STATUS_BAYAR`/`TAG` shifting in the final months are consequences of a customer already having churned, not leading indicators. The leaky `ALL_12` model leans heavily on exactly these columns (see the permutation importance table above), which is why its measured PR-AUC is inflated relative to every window that excludes them. The honest modelling window **`N-11..N-2`** - which drops both contaminated months - is therefore the defensible choice for the rest of this study.

## 6. Verification report

In [7]:
import hashlib
import inspect
import json
import platform

import matplotlib
import sklearn


def _df_to_md_table(frame: pd.DataFrame, float_format: str = "{:.6f}") -> str:
    """Render a small DataFrame as a GitHub-flavored markdown table."""
    header = "| " + " | ".join(frame.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(frame.columns)) + " |"
    lines = [header, sep]
    for _, row in frame.iterrows():
        row_cells = []
        for value in row:
            row_cells.append(float_format.format(value) if isinstance(value, float) else str(value))
        lines.append("| " + " | ".join(row_cells) + " |")
    return "\n".join(lines)


top10_features = set(top10_importances["feature"])

checks = [
    {
        "check": "ALL_12 PR-AUC >= 0.80 AND recall@10% >= 0.85",
        "measured": f"pr_auc={results['ALL_12']['pr_auc']:.6f}, recall@10%={results['ALL_12']['recall_at_10pct']:.6f}",
        "target": "pr_auc >= 0.80 AND recall@10% >= 0.85",
        "passed": bool(results["ALL_12"]["pr_auc"] >= 0.80 and results["ALL_12"]["recall_at_10pct"] >= 0.85),
    },
    {
        "check": "HONEST PR-AUC in [0.33, 0.41] AND recall@10% in [0.45, 0.58]",
        "measured": f"pr_auc={results['HONEST']['pr_auc']:.6f}, recall@10%={results['HONEST']['recall_at_10pct']:.6f}",
        "target": "pr_auc in [0.33, 0.41] AND recall@10% in [0.45, 0.58]",
        "passed": bool(
            0.33 <= results["HONEST"]["pr_auc"] <= 0.41
            and 0.45 <= results["HONEST"]["recall_at_10pct"] <= 0.58
        ),
    },
    {
        "check": "Monotone degradation: PR-AUC(ALL_12) > PR-AUC(DROP_N) > PR-AUC(HONEST)",
        "measured": f"{results['ALL_12']['pr_auc']:.6f} > {results['DROP_N']['pr_auc']:.6f} > {results['HONEST']['pr_auc']:.6f}",
        "target": "strictly decreasing",
        "passed": bool(
            results["ALL_12"]["pr_auc"] > results["DROP_N"]["pr_auc"] > results["HONEST"]["pr_auc"]
        ),
    },
    {
        "check": "Relative PR-AUC inflation (ALL_12 -> HONEST) >= 0.45",
        "measured": f"{relative_inflation:.6f}",
        "target": ">= 0.45",
        "passed": bool(relative_inflation >= 0.45),
    },
    {
        "check": "Leaky-model top-10 features include USAGE_N and >=1 of {TAG_N-1, STATUS_BAYAR_N}",
        "measured": f"top10={sorted(top10_features)}",
        "target": "USAGE_N in top10 AND (TAG_N-1 in top10 OR STATUS_BAYAR_N in top10)",
        "passed": bool(
            "USAGE_N" in top10_features
            and ("TAG_N-1" in top10_features or "STATUS_BAYAR_N" in top10_features)
        ),
    },
]
for c in checks:
    c["result"] = "PASS" if c["passed"] else "FAIL"
    print(f"[{c['result']}] {c['check']} | measured: {c['measured']} | target: {c['target']}")

[PASS] ALL_12 PR-AUC >= 0.80 AND recall@10% >= 0.85 | measured: pr_auc=0.829622, recall@10%=0.925504 | target: pr_auc >= 0.80 AND recall@10% >= 0.85
[PASS] HONEST PR-AUC in [0.33, 0.41] AND recall@10% in [0.45, 0.58] | measured: pr_auc=0.375297, recall@10%=0.506573 | target: pr_auc in [0.33, 0.41] AND recall@10% in [0.45, 0.58]
[PASS] Monotone degradation: PR-AUC(ALL_12) > PR-AUC(DROP_N) > PR-AUC(HONEST) | measured: 0.829622 > 0.521699 > 0.375297 | target: strictly decreasing
[PASS] Relative PR-AUC inflation (ALL_12 -> HONEST) >= 0.45 | measured: 0.547629 | target: >= 0.45
[PASS] Leaky-model top-10 features include USAGE_N and >=1 of {TAG_N-1, STATUS_BAYAR_N} | measured: top10=['PAKET_SPEEDY_ID_11', 'STATUS_BAYAR_N', 'TAG_N', 'TAG_N-1', 'UMUR_PLG', 'USAGE_N', 'USAGE_N-1', 'USAGE_N-3', 'WITEL_3', 'WITEL_5'] | target: USAGE_N in top10 AND (TAG_N-1 in top10 OR STATUS_BAYAR_N in top10)


In [8]:
metrics_source = (PROJECT_ROOT / "src" / "evaluation" / "metrics.py").read_text(encoding="utf-8")
train_eval_source = inspect.getsource(train_and_evaluate_window)

checks_table = pd.DataFrame(
    [{"check": c["check"], "measured": c["measured"], "target": c["target"], "result": c["result"]} for c in checks]
)

report_lines = [
    "# 01 - Leakage-Window Ablation Verification Report",
    "",
    "## Environment",
    f"- Python: {platform.python_version()}",
    f"- numpy: {np.__version__}",
    f"- pandas: {pd.__version__}",
    f"- scikit-learn: {sklearn.__version__}",
    f"- matplotlib: {matplotlib.__version__}",
    f"- Seed: {SEED}",
    "",
    "## Split",
    f"- Method: {split_summary['method']}",
    f"- test_size: {split_summary['test_size']}, random_state: {split_summary['random_state']}",
    f"- n_train: {split_summary['n_train']:,}, n_test: {split_summary['n_test']:,}",
    f"- train positive rate: {split_summary['train_positive_rate']:.6f}",
    f"- test positive rate: {split_summary['test_positive_rate']:.6f}",
    "",
    "## Results table",
    "",
    _df_to_md_table(results_df.reset_index()),
    "",
    "Raw JSON:",
    "```json",
    json.dumps(results, indent=2),
    "```",
    "",
    "## Relative PR-AUC inflation (ALL_12 -> HONEST)",
    f"- {relative_inflation:.6f}",
    "",
    "## Top-10 permutation importances (leaky ALL_12 model)",
    _df_to_md_table(top10_importances),
    "",
    "## Inline source",
    "",
    "### src/evaluation/metrics.py",
    "```python",
    metrics_source,
    "```",
    "",
    "### Core train/evaluate loop (notebook)",
    "```python",
    train_eval_source,
    "```",
    "",
    "## Self-run acceptance checks",
    _df_to_md_table(checks_table, float_format="{}"),
    "",
]
report_text = "\n".join(report_lines)

report_path = OUTPUT_DIR / "01_leakage_ablation_verification.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)

# 01 - Leakage-Window Ablation Verification Report

## Environment
- Python: 3.12.3
- numpy: 2.4.6
- pandas: 3.0.3
- scikit-learn: 1.9.0
- matplotlib: 3.11.0
- Seed: 42

## Split
- Method: stratified train_test_split
- test_size: 0.3, random_state: 42
- n_train: 69,864, n_test: 29,943
- train positive rate: 0.038117
- test positive rate: 0.038106

## Results table

| window | roc_auc | pr_auc | brier | recall_at_10pct |
| --- | --- | --- | --- | --- |
| ALL_12 | 0.974182 | 0.829622 | 0.012922 | 0.925504 |
| DROP_N | 0.881609 | 0.521699 | 0.024402 | 0.651183 |
| HONEST | 0.808064 | 0.375297 | 0.028664 | 0.506573 |
| DROP_N_N1_N2 | 0.788260 | 0.354305 | 0.029214 | 0.489045 |
| FAR | 0.753606 | 0.316389 | 0.030032 | 0.440841 |

Raw JSON:
```json
{
  "ALL_12": {
    "roc_auc": 0.9741821080566941,
    "pr_auc": 0.8296218151760325,
    "brier": 0.012922097472287257,
    "recall_at_10pct": 0.9255039439088519
  },
  "DROP_N": {
    "roc_auc": 0.8816088977899272,
    "pr_auc": 0.521698904246373